# 2. Transform Dataset

*Goal: Parse raw EVTX files into a structured Pandas DataFrame suitable for machine learning.*

## Step 1 — Import Libraries

**In the cell below, you will need to import the required modules for our data transformation:**
 
1. Import the `json` module (we will use this for pretty printing dictionaries).
2. From the standard library `pathlib`, import `Path`, which will be used for handling filesystem paths.
3. From the custom `etl` helper module, import `DocumentTransformer`, `EvtxHandler`, and `Filter`. This will be used for parsing & transforming EVTX data.

## Step 2 — Locate the Dataset

**Next, we need to locate our raw EVTX dataset.**

1. Create a variable called `evtx_path` and assign it a `Path` object pointing to `'../../datasets/EVTX-ATTACK-SAMPLES'`.

*Tip*: you can double check the path exits with the `Path.exists()` function. See the example code snippet below:

Example:
```python
if not evtx_path.exists():
    raise FileNotFoundError("Error Message")
```

## Step 3 — Construct the EVTX Handler class
We have provided a custom module with classes that simplyfy parsing and formatting Windows EVTX records. 

1. Create an `EvtxHandler` object named `evtx_handler` using the `EvtxHandler.from_source` function. Pass the `from_source` function the source path (`evtx_path`) that you defined above.

EVTX records are naturally nested thanks to their XML origins. XML and JSON are not equivilent formats, so its helpful to understand what the raw dictionary versions of records can look like based off the evtx parser being used. 

Let's print the first parsed record to get a better idea. To do this, wrap the `evtx_handler.iterate_records()` in the `next` function to only fetch the first parsed raw record. This will return you a dictionary of the event record, which you can use with `json.dumps(.., indent=2)` to pretty print the record.

## Step 4 — Create a Document Transformer
One of the core phases of any type of data operations is a transformation phase. We need to take raw data and transform it into a format we can work with. Here we will use a custom built class called `DocumentTransformer` from our `etl` module that we will use to pass to our EvtxHandler so it knows how to format documents.

1. Create a variable named `document_transformer`.
2. Assign it the result of `DocumentTransformer.from_fields(<LIST OF TUPLES>)`.

The `DocumentTransformer.from_fields` function takes a list of tuples that define a column name, and a path that maps to the raw record's value. For example, here is a tuple that defines a Timestamp field/column that maps to the raw events SystemTime value: `("Timestamp", "Event.System.TimeCreated.\"#attributes\".SystemTime")`

If we wanted a `DocumentTransformer` that only collects Timestamps from the parsed records, this is what that looks like:

```python
DocumentTransformer.from_fields([
  ("Timestamp", "Event.System.TimeCreated.\"#attributes\".SystemTime")
])
```

Create a DocumentTransformer and assign it to a variable named `document_transformer`. It should contain the following fields/columns:

| Column | Source path |
|---|---|
| `Timestamp` | `Event.System.TimeCreated."#attributes".SystemTime` |
| `Computer` | `Event.System.Computer` |
| `Provider` | `Event.System.Provider."#attributes".Name` |
| `EventID` | `Event.System.EventID."#text"` with fallback to `Event.System.EventID` |
| `CommandLine` | `Event.EventData.CommandLine` |

## Step 5 — Create a Filter

**Windows Event Logs capture a diverse range of record types. Since our goal is to analyze command-line activity, we only want to keep records where a command was actually populated.**

If an event doesn't have a command line associated with it we want to discard it for the purposes of this lab.

1. Create a variable called `evtx_filter` and assign it the result of `Filter.from_pattern()`.
2. Inside the method, you need to provide a string pattern that enforces two rules:
    * The `Event.EventData.CommandLine` field must **not equal** (`!=`) `` `null` ``.
    * **AND** (`&&`) the `Event.EventData.CommandLine` field must **not equal** (`!=`) an empty string (`''`).

Because JMESPath can be a bit of a steep learning curve, below is the pattern string you use to make this happen. Note that in JMESPath, the backtick `` ` `` is used to define special types of variables like null and numeric variables among other things.

```
"Event.EventData.CommandLine != `null` && Event.EventData.CommandLine != ''"
```

**Note** - This is not thorough logic for extracting commands executed from event logs, but it is the logic we will use for workshop purposes.

## Step 6 — Provide EvtxHandler the DocumentTransformer and the Filter

**We need to bring these pieces together into a single, cohesive data processing pipeline (through the `EvtxHandler`!) By attaching our transformer and filter to the source, we create a set of instructions for exactly *where* to get the data, *how* to reshape it, and *what* to keep.**

`evtx_handler` is our `EvtxHandler` object. We need to set the document transformer and the filter that the evtx handler should use for transforming and filtering data. This should be two seperate calls.

1. Pass the `document_transformer` variable to the method `EvtxHandler.with_transformer()` function.
2. Pass the `evtx_filter` variable to the `EvtxHandler.with_filter()` function.

## Step 7 — Parse into a DataFrame

To analyze this data, we need to flatten it into a structured, tabular format (rows and columns). In Python, the gold standard for tabular data is a **Pandas DataFrame**. Now that our pipeline is built, we can execute it and load the transformed results directly into a DataFrame.

1. Execute the `EvtxHandler` by calling the `.parse_into_dataframe()` method on your `evtx_handler` and assign the results to a variable called `dataframe`.
2. We always want to visually inspect our data after parsing. On the last line of the cell, display the first five rows of your DataFrame by using either `dataframe.head()` or `dataframe.iloc[:5]`.

## Step 8 — Summarise the Dataset

Before we save our data and move on to the next phase of the workshop, it is best practice to perform a "sanity check." Did we extract as much data as we expected? Is the data diverse, or is it just the exact same command repeated thousands of times? 

By checking the overall footprint of our dataset, we can validate that our extraction pipeline worked correctly.

Write code to calculate and `print()` the following two statistics:
1. **Total events:** Find the total number of rows in the dataset. *(Hint: You can use `len(dataframe)` or access the first element of `dataframe.shape`)*.
2. **Unique commands:** Find the number of distinct command-line executions. *(Hint: Isolate the `CommandLine` column using bracket notation `["..."]`, then call the `.nunique()` method on it)*.

## Step 9 — Save the DataFrame to Parquet

Persist the transformed DataFrame to disk as a Parquet file named `01_evtx_commands.parquet` using `.to_parquet()`.

> Parquet is a compressed, columnar format that is much faster to read back than CSV and retains column data types — making it ideal for passing data between notebooks in this workshop. We use Parquet over CSV because it can preserve data types which we will need later.